# 📋 [OPS08_M10] 드론 경보 시 관측자-지휘자-경계조 협업 표준운용절차(SOP) 완성
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (정답 & 교관 가이드북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m10_workbook.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M10
- **미션 주제:** 드론 경보 시 관측자-지휘자-경계조 협업 표준운용절차(SOP) 완성
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 10. 드론 경보 시 관측자-지휘자-경계조 협업 표준운용절차(SOP) 완성

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"M1부터 M9까지의 실증 데이터를 종합하여 5개 장(개요-SOP-MSEL-루브릭-AAR) 훈련 패키지와 교관 지침서 최종본을 마크다운으로 편찬해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 타 부대 교관이 별도 교육 없이 즉시 훈련을 통제할 수 있는지, 장별 연계성에 모순이 없는지 최종 승인합니다.

---

### 📑 실습자료 및 과업 지침

드론 경보 시 관측자-지휘자-경계조 협업 표준운용절차(SOP) 완성**

가. 작성 서식 1 - 훈련 패키지 5개 장 구성표

| **장** | **포함 내용** | **반영할 산출물** | **완성(O/X)** |
| --- | --- | --- | --- |
| **1장 개요** | 훈련 목적, 대상, 소요 시간, 가상 상황 | M1 | ( ) |
| **2장 SOP** | 역할별 표준 행동, 흐름도, 요약 카드, 안전 조치 | M2·M8 수정안 | ( ) |
| **3장 시나리오** | MSEL, 오류 주입 카드, 스트레스 타임라인 | M3·M5·M7 | ( ) |
| **4장 평가 루브릭** | SOP 체크, 시각 기록표, 채점표 | M4·M6·M9 | ( ) |
| **5장 AAR** | AAR 진행 순서, 원인 분석 매트릭스, 개선 이력 | M8·M9 | ( ) |

나. 작성 서식 2 - 교관 지침서

| **준비물** | ( ) |
| --- | --- |

| **역할 배정** | ( ) |
| --- | --- |

| **시간표** | ( ) |
| --- | --- |

| **안전 통제** | ( ) |
| --- | --- |

| **채점 방법** | ( ) |
| --- | --- |

| **갱신 주기** | ( ) ※ 연간 교육훈련 계획과 연계 |
| --- | --- |

다. 작성 서식 3 - 성과 수치 일치 확인

| **교안에 적은 수치** | **교안 값** | **원본 기록(M4·M9)** | **일치(O/X)** |
| --- | --- | --- | --- |
| **1차 SOP 준수율** |  |  | ( ) |
| **1차 평균 보고 지연** |  |  | ( ) |
| **재훈련 조치 시간 단축률** |  |  | ( ) |
| **재훈련 오류 감소율** |  |  | ( ) |

라. 작성 서식 4 - 모의 검토 피드백 기록(다른 조 교관 역할)

| **번호** | **검토 의견** | **반영 여부** | **반영 내용** |
| --- | --- | --- | --- |
| **1** |  |  |  |
| **2** |  |  |  |
| **3** |  |  |  |
| **4** |  |  |  |
| **5** |  |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 5개 장이 모두 있다 | ( ) |
| 흐름도·요약 카드 등 시각 자료가 들어갔다 | ( ) |
| 교안 수치가 M4·M9 기록과 일치한다 | ( ) |
| 모의 검토 피드백 반영 내역을 기록했다 | ( ) |
| 실제 부대 정보·실제 음어가 없다 | ( ) |
| 사격은 모의·교전수칙 준수 문구가 있다 | ( ) |

**4. 평가 채점표**

미션마다 정량 기준은 PASS/FAIL로, 정성 기준은 상·중·하로 판정한다. 근거가 되는 기록지·서식 번호를 비고에 적는다.

| **미션** | **구분** | **정량 기준(PASS/FAIL)** | **정성 기준(상·중·하)** | **판정** |
| --- | --- | --- | --- | --- |
| **M1. 야전 부대 드론 경보 상황 분석 및 관측자-지휘자-경계조 역할 정의** | 필수 | 현장 체크 항목 15개 이상 정의, 7단계 전체에 대한 역할 분담 매트릭스 빈칸 0개(100% 매핑), 단계마다 책임자가 1명으로 지정되고 교관 체크리스트 대조에서 누락이 없는가. | 사례 카드의 문제점이 역할 정의에 반영되어 전술 상황이 현실적이고 소부대에 바로 적용 가능한가. | PASS / FAIL
상 / 중 / 하 |
| **M2. 관측자(식별/보고), 지휘자(전파/결심), 경계조(대피/사격) 간 표준 행동 지침서 작성** | 필수 | SOP 7단계 모두에 역할별 세부 절차가 100% 적혔고, 필수 안전/보안 조치 8개가 모두 포함(100%)되었으며, 교차 검토에서 지적된 모호 표현이 모두 수정되었는가. | 긴급 상황에서 한눈에 읽히도록 절차가 명확하고 간결하며, 긴급/일반 모드 구분이 분명한가. | PASS / FAIL
상 / 중 / 하 |
| **M3. 무전 약어 및 육안 식별 보고 템플릿(방위, 고도, 기종, 속도) 표준화** | 필수 | 보고 템플릿에 필수 7항목이 모두 있고, 주입 사건 10건 이상·정상/비정상 6:4 비율을 지켰으며, 워크스루에서 나온 개연성 문제가 모두 수정되었는가. | 상황 전개가 개연성 있고 훈련 내내 적절한 긴장도가 유지되도록 사건 간격과 난이도가 배치되었는가. | PASS / FAIL
상 / 중 / 하 |
| **M4. 가상 드론 경보 발령 시 역할별 초동 조치 및 보고 체계 1차 모의훈련(Dry-run)** | 선택 | SOP 단계 준수율 80% 이상, 최초 포착에서 지휘자 수신까지 보고 지연시간 3분 이내, 병목·혼선 구간 2건 이상 식별했는가. | 현장 판단이 정확하고 보고가 표준 템플릿 순서대로 이루어졌는가. | PASS / FAIL
상 / 중 / 하 |
| **M5. 잘못된 방위 보고 및 통신 지연 등 인위적 오류 주입 시 교차 확인 훈련** | 필수 | 주입된 기만·오경보 식별 성공률 85% 이상, 실제 위협을 오경보로 잘못 버린 건 0건, 판정마다 교차 확인 근거가 기록되었는가. | 의심 신호를 놓치지 않고, 교차 확인 근거가 관측 사실에 기반해 타당한가. | PASS / FAIL
상 / 중 / 하 |
| **M6. 대공 사격 구역 할당 및 인접 분대 상황 전파 인계 실습** | 선택 | 구역 중복·공백 0개, 인계 필수 항목 누락률 0%, 인계 소요 10분 이내, 인수자 퀴즈 5문항 중 4문항 이상 정답인가. | 인계 전후로 상황 인식이 끊기지 않고, 인수자의 질문에 명확히 답했는가. | PASS / FAIL
상 / 중 / 하 |
| **M7. 야간 기습 및 다수 드론 동시 출현 극한 상황 모의 역할분담 스트레스 훈련** | 필수 | 비상 절차 전환 시간 1분 이내, 위험 조치 성공률 90% 이상, 지휘 공백 시 1분 안에 위임이 이루어지고 안전 수칙 위반 0건인가. | 위기 상황에서 우선순위를 냉정하게 판단하고 역할 간 협업이 유지되었는가. | PASS / FAIL
상 / 중 / 하 |
| **M8. AAR 회의를 통한 보고 지연 및 사격 구역 중복 문제점 분석** | 선택 | 개선점 5건 이상, 원인 분석 항목마다 기록지 근거 100% 연결, 회의록에 개인 비난 표현 0건인가. | 비난 없이 사실 중심으로 토의했고 원인 규명이 객관적인가. | PASS / FAIL
상 / 중 / 하 |
| **M9. 피드백 반영 역할별 보고 및 사격 절차 재훈련을 통한 반응 속도 개선** | 필수 | 1차 대비 평균 조치 시간 30% 이상 단축, 오류 발생 건수 50% 이상 감소, 동일 평가표·동일 측정 방법을 사용했는가. | 행동 절차가 몸에 익어 역할 간 호흡이 맞고 말없이도 다음 행동이 이어지는가. | PASS / FAIL
상 / 중 / 하 |
| **M10. 드론 경보 시 관측자-지휘자-경계조 협업 표준운용절차(SOP) 완성** | 필수 | 5개 장 100% 완비, 교안에 적힌 성과 수치가 M4·M9 기록과 일치하고, 모의 검토 피드백 반영 내역이 기록되었는가. | 교재가 완성도 있고 시각 자료가 이해를 돕으며 다른 부대에 전파할 가치가 있는가. | PASS / FAIL
상 / 중 / 하 |

| 이수 기준(권장): 필수 미션 7개가 모두 PASS이고 정성 판정에 '하'가 2개 이하일 것. 선택 미션(M4·M6·M8)은 수행 시 가점으로 반영하며 세부 배점은 교관이 정한다. |
| --- |

| **평가관** | 소속·성명: ( ) 서명: ( ) 일자: ( ) |
| --- | --- |

| **종합 의견** | ( ) |
| --- | --- |



In [ ]:
# [미션 10 실습 코드] 최종 5개 장 훈련 패키지 마크다운 자동 Export
import os
os.makedirs('package', exist_ok=True)
package_path = 'package/ops08_standard_training_package.md'
with open(package_path, 'w', encoding='utf-8') as f:
    f.write('# [OPS08] 소부대 드론 경보 협업 표준운용절차(SOP) 및 표준 훈련 패키지\n')
    f.write('## 제1장 개요 | 제2장 SOP | 제3장 MSEL 시나리오 | 제4장 평가루브릭 | 제5장 AAR 가이드\n\n')
    f.write('본 문서는 M1~M9 훈련 실증 데이터를 집대성한 전군 야전부대 배포용 표준 패키지입니다.\n')
print(f'✔ 최종 훈련 패키지 파일 내보내기 완료: {package_path}')
